# LeetCode 328 — Odd Even Linked List bằng Python

Link đề: https://leetcode.com/problems/odd-even-linked-list/ · Độ khó: **Medium** · Chủ đề: Linked List

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def oddEvenList(self, head: Optional[ListNode]) -> Optional[ListNode]:
```

Ghi chú: LeetCode tự cung cấp `ListNode`; trong notebook ta tự dựng `ListNode` và hai hàm đổi **list ↔ linked list** để test.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: gom các nút ở **vị trí lẻ** (nút thứ 1, 3, 5, ...) đứng trước, rồi đến các nút ở **vị trí chẵn** (nút thứ 2, 4, ...), giữ nguyên **thứ tự tương đối** trong mỗi nhóm; yêu cầu **`O(1)` bộ nhớ phụ**, **`O(n)` thời gian**.
2. Chú ý: "lẻ/chẵn" là theo **chỉ số vị trí** (bắt đầu từ 1), **không phải giá trị** của nút.
3. Đi từ **chép giá trị ra mảng rồi ghi lại** → **gom nút vào hai danh sách rồi nối** → **hai con trỏ `odd`/`even` tách rồi nối** `O(1)` phụ → **đệ quy**.
4. Hiểu **vì sao phải giữ `even_head`** và **vì sao điều kiện lặp là `even and even.next`**.
5. Biết kiểm tra bài linked list bằng cách **dựng danh sách từ list**, **chuyển ngược lại để so sánh**, và kiểm tra **các nút vẫn là chính những nút cũ** (chỉ đổi liên kết).
6. Tránh các lỗi kinh điển: mất `even_head`, quên nối hai nhóm, lặp sai điều kiện dừng, nhầm "chỉ số" với "giá trị".

## 2. Đề bài & ví dụ

Cho `head` của một danh sách liên kết đơn, hãy gom tất cả các nút có chỉ số **lẻ** lại với nhau, theo sau là các nút có chỉ số **chẵn**, và trả về danh sách đã sắp xếp lại.

Nút **đầu tiên** được coi là **lẻ**, nút thứ hai là **chẵn**, v.v. Thứ tự tương đối trong mỗi nhóm phải giữ nguyên như đầu vào. Bạn phải giải bằng **`O(1)`** bộ nhớ phụ và **`O(n)`** thời gian.

**Ví dụ 1**

- Input: `head = [1,2,3,4,5]`
- Output: `[1,3,5,2,4]`

**Ví dụ 2**

- Input: `head = [2,1,3,5,6,4,7]`
- Output: `[2,3,6,7,1,5,4]`

**Ràng buộc:** số nút trong khoảng `[0, 10^4]`; `-10^6 <= Node.val <= 10^6`.

In [1]:
from typing import Optional, List
import random
import sys
import time


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    """list -> linked list, trả về head (None nếu rỗng)."""
    dummy = ListNode()
    cur = dummy
    for v in values:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next


def to_list(head, limit=100_000):
    """linked list -> list (có giới hạn bước để không treo nếu lỡ có chu trình)."""
    out = []
    while head is not None and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out


def node_ids(head, limit=100_000):
    out = []
    while head is not None and len(out) < limit:
        out.append(id(head))
        head = head.next
    return out


examples = [
    ([1, 2, 3, 4, 5], [1, 3, 5, 2, 4]),
    ([2, 1, 3, 5, 6, 4, 7], [2, 3, 6, 7, 1, 5, 4]),
]
for values, expected in examples:
    print(values, "->", expected, "| to_list(build_list(...)) =", to_list(build_list(values)))

[1, 2, 3, 4, 5] -> [1, 3, 5, 2, 4] | to_list(build_list(...)) = [1, 2, 3, 4, 5]
[2, 1, 3, 5, 6, 4, 7] -> [2, 3, 6, 7, 1, 5, 4] | to_list(build_list(...)) = [2, 1, 3, 5, 6, 4, 7]


## 3. Kiến thức nền

### 3.1. Danh sách liên kết đơn

Mỗi nút có `val` và `next` (nút kế tiếp hoặc `None`). Ta chỉ đi **một chiều** và chỉ đổi **liên kết** `next`. Đổi thứ tự nút chính là gán lại `next`.
#### Code demo

In [2]:
head = build_list([1, 2, 3])
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### 3.2. Tách một danh sách thành hai danh sách bằng hai con trỏ

Nếu `odd` trỏ tới nút lẻ và `even = odd.next` là nút chẵn kế bên, thì "bỏ qua nút chẵn" là `odd.next = even.next`, và "bỏ qua nút lẻ kế tiếp" là `even.next = odd.next.next` (sau khi `odd` đã nhảy). Lặp lại sẽ tách chuỗi thành hai chuỗi xen kẽ **mà không tạo nút mới**.
#### Code demo

In [3]:
head = build_list([1, 2, 3, 4, 5])
odd, even = head, head.next
even_head = even
while even and even.next:
    odd.next = even.next
    odd = odd.next
    even.next = odd.next
    even = even.next
print("chuỗi lẻ:", to_list(head), "| chuỗi chẵn:", to_list(even_head))

chuỗi lẻ: [1, 3, 5] | chuỗi chẵn: [2, 4]


### 3.3. Nối đuôi: `odd.next = even_head`

Sau khi tách, `odd` đang ở **nút lẻ cuối cùng**; nối `odd.next = even_head` để chuỗi lẻ đi tiếp vào chuỗi chẵn. Nút chẵn cuối cùng đã có `next = None` nên danh sách kết thúc đúng.
#### Code demo

In [4]:
odd.next = even_head
print(to_list(head))

[1, 3, 5, 2, 4]


### 3.4. Test trên linked list: giá trị và danh tính nút

Hai kiểm tra: (1) `to_list(kết quả)` bằng danh sách mong đợi; (2) **danh tính nút**: các nút phải là **chính những nút cũ** (so sánh `id`), chứng tỏ ta chỉ đổi liên kết, không tạo nút mới hay chép giá trị.
#### Code demo

In [5]:
head = build_list([10, 20, 30, 40])
before = node_ids(head)
print("số nút:", len(before), "| id nút lẻ (vị trí 1, 3):", len(before[0::2]), "nút; id nút chẵn (vị trí 2, 4):", len(before[1::2]), "nút")
print("thứ tự id mong đợi sau khi gom = ids[0::2] + ids[1::2] (cùng các nút cũ, chỉ đổi liên kết)")

số nút: 4 | id nút lẻ (vị trí 1, 3): 2 nút; id nút chẵn (vị trí 2, 4): 2 nút
thứ tự id mong đợi sau khi gom = ids[0::2] + ids[1::2] (cùng các nút cũ, chỉ đổi liên kết)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`[2, 1, 3, 5, 6, 4, 7]`: vị trí (từ 1): `2(1) 1(2) 3(3) 5(4) 6(5) 4(6) 7(7)`.

- Vị trí lẻ (1,3,5,7): `2, 3, 6, 7`.
- Vị trí chẵn (2,4,6): `1, 5, 4`.
- Ghép: `[2, 3, 6, 7, 1, 5, 4]` — khớp ví dụ 2.

Nhận xét quan trọng: nhóm lẻ là `vals[0::2]`, nhóm chẵn là `vals[1::2]`; đáp án = `vals[0::2] + vals[1::2]`. Với linked list ta muốn làm đúng điều này **bằng cách nối lại con trỏ**, không dùng mảng phụ.

## 5. Cách 1 — Brute Force / baseline: chép giá trị ra mảng rồi ghi lại

**Ý tưởng.** Duyệt danh sách, chép các `val` vào một mảng; tính `vals[0::2] + vals[1::2]`; duyệt lần nữa ghi đè `val` của từng nút theo thứ tự mới. Đơn giản nhưng dùng `O(n)` bộ nhớ và **thay đổi giá trị trong nút** (không phải đổi liên kết) — vi phạm yêu cầu `O(1)` phụ.
#### Code demo

In [6]:
def odd_even_copy_values(head):
    vals = to_list(head)
    new_vals = vals[0::2] + vals[1::2]
    cur = head
    for v in new_vals:
        cur.val = v
        cur = cur.next
    return head


for values, expected in examples:
    print(values, "->", to_list(odd_even_copy_values(build_list(values))), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> [1, 3, 5, 2, 4] | kỳ vọng: [1, 3, 5, 2, 4]
[2, 1, 3, 5, 6, 4, 7] -> [2, 3, 6, 7, 1, 5, 4] | kỳ vọng: [2, 3, 6, 7, 1, 5, 4]


**Độ phức tạp (Cách 1).** **Time `O(n)`**, **Space `O(n)`**.

**Nút thắt.** Dùng mảng phụ và sửa `val`; ta muốn chỉ sửa `next` và dùng `O(1)` phụ.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ hai con trỏ: `odd` (đang ở nút lẻ cuối của chuỗi lẻ đã dựng) và `even` (đang ở nút chẵn cuối của chuỗi chẵn đã dựng), cộng `even_head` để nhớ **đầu chuỗi chẵn**.

Mỗi vòng lặp:

1. `odd.next = even.next` — nút lẻ nối thẳng tới nút lẻ kế tiếp (bỏ qua nút chẵn).
2. `odd = odd.next`.
3. `even.next = odd.next` — nút chẵn nối tới nút chẵn kế tiếp.
4. `even = even.next`.

Dừng khi `even` hoặc `even.next` là `None`. Cuối cùng nối `odd.next = even_head`. Mỗi nút bị chạm một lần: `O(n)` thời gian, chỉ vài con trỏ: `O(1)` phụ.

## 7. Các cách tiếp theo

### Cách 2 — Gom nút vào hai danh sách (Python list) rồi nối lại

**Giải thích.** Duyệt, đưa các **nút** (không phải giá trị) có chỉ số chẵn/lẻ vào `odds` và `evens`; sau đó nối `odds[0] → ... → odds[-1] → evens[0] → ... → evens[-1] → None`. Chỉ đổi liên kết, nhưng dùng `O(n)` bộ nhớ cho hai danh sách nút.
#### Code demo

In [7]:
def odd_even_node_lists(head):
    if head is None:
        return None
    odds, evens = [], []
    cur, idx = head, 1
    while cur:
        (odds if idx % 2 == 1 else evens).append(cur)
        cur = cur.next
        idx += 1
    chain = odds + evens
    for a, b in zip(chain, chain[1:]):
        a.next = b
    chain[-1].next = None
    return chain[0]


for values, expected in examples:
    print(values, "->", to_list(odd_even_node_lists(build_list(values))), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> [1, 3, 5, 2, 4] | kỳ vọng: [1, 3, 5, 2, 4]
[2, 1, 3, 5, 6, 4, 7] -> [2, 3, 6, 7, 1, 5, 4] | kỳ vọng: [2, 3, 6, 7, 1, 5, 4]


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (danh sách các nút).

### Cách 3 — Hai con trỏ `odd`/`even` tách rồi nối (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
def odd_even_two_pointers(head):
    if head is None or head.next is None:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head


for values, expected in examples:
    print(values, "->", to_list(odd_even_two_pointers(build_list(values))), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> [1, 3, 5, 2, 4] | kỳ vọng: [1, 3, 5, 2, 4]
[2, 1, 3, 5, 6, 4, 7] -> [2, 3, 6, 7, 1, 5, 4] | kỳ vọng: [2, 3, 6, 7, 1, 5, 4]


In [9]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def oddEvenList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        if head is None or head.next is None:
            return head
        odd, even = head, head.next
        even_head = even                       # nhớ đầu chuỗi chẵn
        while even and even.next:
            odd.next = even.next
            odd = odd.next
            even.next = odd.next
            even = even.next
        odd.next = even_head                   # nối chuỗi lẻ vào chuỗi chẵn
        return head


print(to_list(Solution().oddEvenList(build_list([1, 2, 3, 4, 5]))))

[1, 3, 5, 2, 4]


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Đệ quy

**Giải thích.** Hàm `go(odd, even)` thực hiện đúng một bước của Cách 3 rồi gọi lại cho cặp tiếp theo; khi hết nút thì nối `odd.next = even_head`. Đây là cách viết lại vòng lặp bằng đệ quy: ngắn gọn nhưng dùng **`O(n)` ngăn xếp** (độ sâu `~ n/2`).
#### Code demo

In [10]:
sys.setrecursionlimit(50_000)


def odd_even_recursive(head):
    if head is None or head.next is None:
        return head
    even_head = head.next

    def go(odd, even):
        if even is None or even.next is None:
            odd.next = even_head
            return
        nxt_odd = even.next
        odd.next = nxt_odd
        even.next = nxt_odd.next
        go(nxt_odd, even.next)

    go(head, even_head)
    return head


for values, expected in examples:
    print(values, "->", to_list(odd_even_recursive(build_list(values))), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> [1, 3, 5, 2, 4] | kỳ vọng: [1, 3, 5, 2, 4]
[2, 1, 3, 5, 6, 4, 7] -> [2, 3, 6, 7, 1, 5, 4] | kỳ vọng: [2, 3, 6, 7, 1, 5, 4]


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (ngăn xếp đệ quy, độ sâu ~ `n/2`; `n ≤ 10^4` nên cần nâng giới hạn đệ quy).

> Cách 3 là bản nộp chuẩn và duy nhất đạt `O(1)` phụ + chỉ đổi liên kết. Cách 1 vi phạm `O(1)`; Cách 2 vi phạm `O(1)`; Cách 4 vi phạm `O(1)` vì ngăn xếp.

## 8. TRACE từng bước

Ta trace Cách 3 trên `[1, 2, 3, 4, 5]` và `[1, 2, 3, 4]`: in `odd`, `even` và hai chuỗi hiện có (chuỗi lẻ từ `head`, chuỗi chẵn từ `even_head`).
#### Code demo

In [11]:
def trace_odd_even(values):
    head = build_list(values)
    print("đầu vào:", values)
    if head is None or head.next is None:
        print("  <= 1 nút: trả nguyên")
        return head
    odd, even = head, head.next
    even_head = even
    step = 0

    def show(tag):
        # in chuỗi lẻ đến `odd` và chuỗi chẵn đến `even` (an toàn, không đi qua chỗ chưa nối)
        o, chain_odd = head, []
        while True:
            chain_odd.append(o.val)
            if o is odd:
                break
            o = o.next
        e, chain_even = even_head, []
        while e is not None:
            chain_even.append(e.val)
            if e is even:
                break
            e = e.next
        print(f"  {tag:<12} odd={odd.val} even={even.val if even else None} | lẻ: {chain_odd} | chẵn: {chain_even}")

    show("khởi tạo")
    while even and even.next:
        step += 1
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
        show(f"sau bước {step}")
    odd.next = even_head
    print(f"  nối odd.next = even_head => {to_list(head)}")
    return head


trace_odd_even([1, 2, 3, 4, 5])
print()
trace_odd_even([1, 2, 3, 4])

đầu vào: [1, 2, 3, 4, 5]
  khởi tạo     odd=1 even=2 | lẻ: [1] | chẵn: [2]
  sau bước 1   odd=3 even=4 | lẻ: [1, 3] | chẵn: [2, 4]
  sau bước 2   odd=5 even=None | lẻ: [1, 3, 5] | chẵn: [2, 4]
  nối odd.next = even_head => [1, 3, 5, 2, 4]

đầu vào: [1, 2, 3, 4]
  khởi tạo     odd=1 even=2 | lẻ: [1] | chẵn: [2]
  sau bước 1   odd=3 even=4 | lẻ: [1, 3] | chẵn: [2, 4]
  nối odd.next = even_head => [1, 3, 2, 4]


Bảng tóm tắt (`[1, 2, 3, 4, 5]`):

| bước | `odd` | `even` | chuỗi lẻ | chuỗi chẵn |
|---|---|---|---|---|
| khởi tạo | 1 | 2 | `[1]` | `[2]` |
| 1 | 3 | 4 | `[1, 3]` | `[2, 4]` |
| 2 | 5 | `None` (dừng vì `even` rỗng) | `[1, 3, 5]` | `[2, 4]` |
| nối | `5.next = 2` | | `[1, 3, 5, 2, 4]` | |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Đúng `class Solution`, đúng `oddEvenList(self, head: Optional[ListNode]) -> Optional[ListNode]`.
#### Code demo

In [12]:
class Solution:
    def oddEvenList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        if head is None or head.next is None:      # 0 hoặc 1 nút
            return head
        odd, even = head, head.next                # nút lẻ đầu, nút chẵn đầu
        even_head = even                           # nhớ đầu chuỗi chẵn để nối sau cùng
        while even and even.next:
            odd.next = even.next                   # lẻ -> lẻ kế tiếp
            odd = odd.next
            even.next = odd.next                   # chẵn -> chẵn kế tiếp
            even = even.next
        odd.next = even_head                       # nối đuôi chuỗi lẻ vào đầu chuỗi chẵn
        return head


sol = Solution()
for values, expected in examples:
    assert to_list(sol.oddEvenList(build_list(values))) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

Bất biến vòng lặp: **trước mỗi lần lặp**, `head ... odd` là chuỗi lẻ (đã nối đúng thứ tự), `even_head ... even` là chuỗi chẵn (đã nối đúng thứ tự), `odd.next = even` (nút chẵn đang chờ) và `even.next` là nút lẻ kế tiếp chưa xử lý (hoặc `None`).

- **Khởi tạo:** `odd = nút 1`, `even = nút 2`: chuỗi lẻ `[1]`, chuỗi chẵn `[2]`.
- **Bảo toàn:** `odd.next = even.next` đưa nút lẻ kế tiếp vào chuỗi lẻ; `odd = odd.next`; lúc đó `odd.next` là nút chẵn kế tiếp; `even.next = odd.next` đưa nó vào chuỗi chẵn; `even = even.next`. Mỗi nút được thêm đúng một lần vào đúng chuỗi, **giữ thứ tự tương đối**.
- **Dừng:** `even is None` (số nút lẻ → nút cuối là lẻ, `odd` là nút cuối) hoặc `even.next is None` (số nút chẵn → nút cuối là chẵn, `even` là nút cuối). Cả hai trường hợp `odd` đều là nút lẻ cuối và `even.next` là `None` nên chuỗi chẵn đã kết thúc đúng.
- **Nối:** `odd.next = even_head` ghép hai chuỗi; vì `even.next = None` nên danh sách kết thúc và **không có chu trình**.
- **Giữ `even_head`:** sau lần tách đầu, nút 2 không còn được trỏ tới bởi ai; nếu mất `even_head` ta mất cả chuỗi chẵn.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút (≤ `10^4`).

| Cách | Time | Space |
|---|---|---|
| 1. Chép giá trị rồi ghi lại | `O(n)` | `O(n)` |
| 2. Gom nút vào hai list | `O(n)` | `O(n)` |
| 3. Hai con trỏ | `O(n)` | `O(1)` |
| 4. Đệ quy | `O(n)` | `O(n)` ngăn xếp |

Mỗi nút bị chạm hằng số lần (một lần gán `next`, một lần di chuyển con trỏ). Với `n = 10^4`, mọi cách đều chạy trong vài mili-giây.

## 12. Bộ test

`oracle(values)` = `values[0::2] + values[1::2]` — một công thức trên **mảng**, **độc lập** với việc nối con trỏ. Mỗi lần test ta dựng danh sách mới (vì thuật toán sửa tại chỗ).

Hàm `check(fn, values, same_nodes=True)`: (1) so giá trị với `oracle`; (2) nếu `same_nodes`, kiểm tra các nút vẫn là **chính những nút cũ** theo đúng thứ tự `ids[0::2] + ids[1::2]`; (3) kiểm tra không có chu trình (độ dài đúng bằng `n`).
#### Code demo

In [13]:
def oracle(values):
    return values[0::2] + values[1::2]


def check(fn, values, same_nodes=True):
    head = build_list(values)
    ids = node_ids(head)
    result = fn(head)
    got = to_list(result, limit=len(values) + 5)
    assert got == oracle(values), (fn.__name__, values, got, oracle(values))
    assert len(got) == len(values), "độ dài sai hoặc có chu trình"
    if same_nodes:
        assert node_ids(result, limit=len(values) + 5) == ids[0::2] + ids[1::2], "không giữ danh tính nút"


impls = {
    "copy_values": (odd_even_copy_values, False),      # không đổi liên kết, chỉ chép giá trị
    "node_lists": (odd_even_node_lists, True),
    "two_pointers": (odd_even_two_pointers, True),
    "recursive": (odd_even_recursive, True),
    "submit": (sol.oddEvenList, True),
}

for values, expected in examples:
    for key, (fn, same) in impls.items():
        check(fn, values, same)
        assert to_list(fn(build_list(values))) == expected
print("PASS ví dụ của đề")

PASS ví dụ của đề


### 12.1. Các trường hợp biên
#### Code demo

In [14]:
edge_cases = [
    ([], "danh sách rỗng"),
    ([7], "một nút"),
    ([1, 2], "hai nút"),
    ([1, 2, 3], "ba nút"),
    ([1, 2, 3, 4], "bốn nút (chẵn)"),
    ([5, 5, 5, 5, 5], "mọi giá trị bằng nhau (danh tính nút vẫn phân biệt)"),
    ([-10 ** 6, 10 ** 6, -10 ** 6, 10 ** 6], "giá trị biên"),
    ([0, 0, 0], "toàn số 0"),
    (list(range(1, 11)), "1..10"),
    (list(range(10, 0, -1)), "10..1"),
]
for values, label in edge_cases:
    for key, (fn, same) in impls.items():
        check(fn, values, same)
    print("PASS", label)

PASS danh sách rỗng
PASS một nút
PASS hai nút
PASS ba nút
PASS bốn nút (chẵn)
PASS mọi giá trị bằng nhau (danh tính nút vẫn phân biệt)
PASS giá trị biên
PASS toàn số 0
PASS 1..10
PASS 10..1


### 12.2. Tính chất và phản ví dụ

- Độ dài và đa tập giá trị không đổi.
- Phần tử đầu luôn giữ nguyên (nút lẻ đầu tiên vẫn đứng đầu).
- Nhóm lẻ giữ nguyên thứ tự tương đối, nhóm chẵn giữ nguyên thứ tự tương đối.
- **Phản ví dụ** cho "lẻ/chẵn theo giá trị": `[2, 1, 3, 5, 6, 4, 7]`.
#### Code demo

In [15]:
rng = random.Random(328)
for _ in range(300):
    n = rng.randint(0, 20)
    values = [rng.randint(-5, 5) for _ in range(n)]
    out = to_list(sol.oddEvenList(build_list(values)))
    assert sorted(out) == sorted(values) and len(out) == n
    if n:
        assert out[0] == values[0]
        assert out[: (n + 1) // 2] == values[0::2] and out[(n + 1) // 2:] == values[1::2]
print("PASS: đa tập giá trị, phần tử đầu, thứ tự tương đối trong từng nhóm")


def sai_theo_gia_tri(values):
    odds = [v for v in values if v % 2 != 0]               # SAI: lẻ/chẵn theo GIÁ TRỊ
    evens = [v for v in values if v % 2 == 0]
    return odds + evens


bad = [2, 1, 3, 5, 6, 4, 7]
print("theo giá trị:", sai_theo_gia_tri(bad), "| đúng theo vị trí:", oracle(bad))
assert sai_theo_gia_tri(bad) != oracle(bad)

PASS: đa tập giá trị, phần tử đầu, thứ tự tương đối trong từng nhóm
theo giá trị: [1, 3, 5, 7, 2, 6, 4] | đúng theo vị trí: [2, 3, 6, 7, 1, 5, 4]


### 12.3. Kiểm tra ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [16]:
rng = random.Random(2328)
for trial in range(2000):
    n = rng.randint(0, 25)
    values = [rng.randint(-10 ** 6, 10 ** 6) for _ in range(n)]
    for key, (fn, same) in impls.items():
        check(fn, values, same)
print("PASS 2000 test ngẫu nhiên (giá trị + danh tính nút + không chu trình)")

# n = 10^4 (và các kích thước sát biên)
for n in (9999, 10_000):
    values = [rng.randint(-10 ** 6, 10 ** 6) for _ in range(n)]
    row = []
    for key, (fn, same) in impls.items():
        head = build_list(values)
        t0 = time.perf_counter()
        result = fn(head)
        dt = (time.perf_counter() - t0) * 1000
        assert to_list(result, limit=n + 5) == oracle(values)
        row.append(f"{key}={dt:6.2f}ms")
    print(f"n={n:<6} | " + " ".join(row))
print("PASS n = 10^4")

PASS 2000 test ngẫu nhiên (giá trị + danh tính nút + không chu trình)


n=9999   | copy_values=  1.33ms node_lists=  2.53ms two_pointers=  0.86ms recursive=  1.79ms submit=  0.81ms


n=10000  | copy_values=  1.57ms node_lists=  1.85ms two_pointers=  0.84ms recursive=  2.05ms submit=  0.58ms
PASS n = 10^4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không giữ `even_head`
#### Code sai

In [17]:
def sai_mat_even_head(head):
    if head is None or head.next is None:
        return head
    odd, even = head, head.next                  # SAI: không lưu even_head
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = None                              # không còn biết đầu chuỗi chẵn để nối
    return head


print("sai:", to_list(sai_mat_even_head(build_list([1, 2, 3, 4, 5]))), "| đúng:", to_list(sol.oddEvenList(build_list([1, 2, 3, 4, 5]))))

sai: [1, 3, 5] | đúng: [1, 3, 5, 2, 4]


**Vì sao sai:** sau khi tách, không ai còn trỏ tới nút `2` (đầu chuỗi chẵn); mất tham chiếu ⇒ mất toàn bộ chuỗi chẵn. **Code đúng:** `even_head = head.next` ngay từ đầu.

### Lỗi 2: Quên nối chuỗi lẻ vào chuỗi chẵn
#### Code sai

In [18]:
def sai_quen_noi(head):
    if head is None or head.next is None:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    return head                                  # SAI: thiếu odd.next = even_head


print("sai:", to_list(sai_quen_noi(build_list([1, 2, 3, 4, 5]))), "| đúng:", to_list(sol.oddEvenList(build_list([1, 2, 3, 4, 5]))))

sai: [1, 3, 5] | đúng: [1, 3, 5, 2, 4]


**Vì sao sai:** hai chuỗi vẫn tách rời, kết quả chỉ chứa nhóm lẻ. **Code đúng:** `odd.next = even_head` trước khi trả về.

### Lỗi 3: Điều kiện lặp `while even.next` (thiếu kiểm tra `even`)
#### Code sai

In [19]:
def sai_dieu_kien(head):
    if head is None or head.next is None:
        return head
    odd, even = head, head.next
    even_head = even
    while even.next:                             # SAI: even có thể là None sau một bước (n lẻ)
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head


for values in ([1, 2, 3, 4], [1, 2, 3, 4, 5]):
    try:
        print(values, "->", to_list(sai_dieu_kien(build_list(values))))
    except AttributeError as e:
        print(values, "-> AttributeError:", e)

[1, 2, 3, 4] -> [1, 3, 2, 4]
[1, 2, 3, 4, 5] -> AttributeError: 'NoneType' object has no attribute 'next'


**Vì sao sai:** với số nút **lẻ**, sau vòng cuối `even` trở thành `None`, `even.next` gây `AttributeError`. **Code đúng:** `while even and even.next`.

### Lỗi 4: Phân nhóm theo giá trị chứ không theo vị trí
Xem `sai_theo_gia_tri` ở mục 12.2: `[2, 1, 3, 5, 6, 4, 7]` cho `[1, 3, 5, 7, 2, 6, 4]` thay vì `[2, 3, 6, 7, 1, 5, 4]`. **Vì sao sai:** đề nói **chỉ số** (vị trí), không phải tính chẵn lẻ của giá trị. **Code đúng:** đếm vị trí hoặc dùng cặp con trỏ nhảy cách một nút.

### Lỗi 5: Làm đảo thứ tự trong nhóm chẵn
#### Code sai

In [20]:
def sai_dao_nhom_chan(head):
    if head is None or head.next is None:
        return head
    odd = head
    rest = head.next
    rev_even = None                              # SAI: chèn nút chẵn vào ĐẦU chuỗi -> đảo ngược
    cur, idx = rest, 2
    odd_tail = head
    while cur:
        nxt = cur.next
        if idx % 2 == 0:
            cur.next = rev_even
            rev_even = cur
        else:
            odd_tail.next = cur
            odd_tail = cur
        cur = nxt
        idx += 1
    odd_tail.next = rev_even
    return head


print("sai:", to_list(sai_dao_nhom_chan(build_list([1, 2, 3, 4, 5, 6, 7]))), "| đúng:", to_list(sol.oddEvenList(build_list([1, 2, 3, 4, 5, 6, 7]))))

sai: [1, 3, 5, 7, 6, 4, 2] | đúng: [1, 3, 5, 7, 2, 4, 6]


**Vì sao sai:** đề yêu cầu **giữ nguyên thứ tự tương đối** trong mỗi nhóm; chèn đầu làm nhóm chẵn bị đảo ngược. **Code đúng:** thêm nút vào **cuối** chuỗi (như `even.next = ...; even = even.next`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**gom nút theo vị trí chẵn/lẻ**, `O(1)` phụ" → **hai con trỏ `odd`/`even` + `even_head`**.
- Điều kiện dừng chuẩn: **`while even and even.next`**.
- **Luôn lưu `even_head`** trước khi tách; **nối `odd.next = even_head`** sau cùng.
- Test linked list: dựng từ list, chuyển ngược về list; kiểm tra **danh tính nút** và **không chu trình**.
- Họ hàng: **Partition List** (86, tách theo giá trị `< x` và `≥ x` bằng hai danh sách giả đầu), **Reorder List** (143), **Sort List** (148), **Swap Nodes in Pairs** (24), **Reverse Linked List** (206).
- Kỹ thuật **dummy head** giúp bỏ các trường hợp đặc biệt khi dựng danh sách mới.
- Thử: `[]`, `[7]`, `[1,2]`, `[1,2,3]`, `[1,2,3,4]`.

## 15. Biến thể của bài

1. **Partition List** (LeetCode 86): tách theo **giá trị** (`< x` trước, `≥ x` sau) bằng hai danh sách giả đầu, giữ thứ tự.
2. **Gom theo `k` nhóm vị trí** (vị trí `≡ 1, 2, ..., k` mod `k`): tổng quát hóa lẻ/chẵn.
3. **Đảo thứ tự**: nhóm chẵn đứng trước nhóm lẻ.
4. **Odd Even với giá trị**: gom nút có **giá trị** lẻ trước, giữ thứ tự (phân biệt với bài này).
5. **Xen kẽ lại** (hoàn tác): từ `[1,3,5,2,4]` về `[1,2,3,4,5]` (Reorder-like).
#### Code demo

In [21]:
def partition_list(head, x):
    """LeetCode 86 (dùng dummy head): nút < x trước, nút >= x sau, giữ thứ tự."""
    small_dummy, big_dummy = ListNode(), ListNode()
    s, b = small_dummy, big_dummy
    while head:
        if head.val < x:
            s.next = head
            s = s.next
        else:
            b.next = head
            b = b.next
        head = head.next
    b.next = None
    s.next = big_dummy.next
    return small_dummy.next


def group_mod_k(head, k):
    """Gom theo vị trí mod k: nhóm 0 (vị trí 1, k+1, ...) rồi nhóm 1, ... nhóm k-1."""
    heads = [ListNode() for _ in range(k)]
    tails = list(heads)
    idx = 0
    while head:
        nxt = head.next
        tails[idx % k].next = head
        tails[idx % k] = head
        head.next = None
        head = nxt
        idx += 1
    cur = ListNode()                                    # nối các nhóm (có thể có nhóm rỗng)
    start = cur
    for g in range(k):
        cur.next = heads[g].next
        while cur.next:
            cur = cur.next
    return start.next


print(to_list(partition_list(build_list([1, 4, 3, 2, 5, 2]), 3)))      # [1, 2, 2, 4, 3, 5]
print(to_list(group_mod_k(build_list([1, 2, 3, 4, 5, 6, 7]), 3)))      # [1, 4, 7, 2, 5, 3, 6]
print(to_list(group_mod_k(build_list([1, 2, 3, 4, 5]), 2)))            # [1, 3, 5, 2, 4] (bài này khi k = 2)

[1, 2, 2, 4, 3, 5]
[1, 4, 7, 2, 5, 3, 6]
[1, 3, 5, 2, 4]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chép giá trị | mảng phụ, ghi đè `val` | `O(n)` | `O(n)` | baseline, sai yêu cầu `O(1)` |
| 2. Hai list nút | gom nút, nối lại | `O(n)` | `O(n)` | dễ hiểu, dễ viết |
| 3. Hai con trỏ | `odd`/`even` + `even_head` | `O(n)` | `O(1)` | **bản nộp chuẩn** |
| 4. Đệ quy | một bước mỗi cuộc gọi | `O(n)` | `O(n)` | minh họa |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `oracle([1,2,3,4,5,6])`; (b) `oracle([9])`; (c) `oracle([4,3])`; (d) hai nhóm của `[7,8,9,10,11]`.

**Bài 2 (tính bằng tay).** Chạy Cách 3 cho `[10, 20, 30, 40]`: ghi `odd`, `even` sau mỗi bước; đến lúc dừng, nối đuôi.

**Bài 3 (điền TODO).** Hoàn thiện `odd_even_todo(head)`.
#### Code demo

In [22]:
def odd_even_todo(head):
    # TODO 1: nếu head rỗng hoặc chỉ 1 nút thì trả head
    # TODO 2: odd, even = head, head.next; even_head = even
    # TODO 3: while even and even.next: nối odd tới nút lẻ kế, even tới nút chẵn kế
    # TODO 4: odd.next = even_head; return head
    pass


# Tự làm xong rồi mới mở test:
# for values, expected in examples: assert to_list(odd_even_todo(build_list(values))) == expected

**Bài 4 (tự viết lại).** Viết `even_odd_list(head)` đưa nhóm **chẵn trước, lẻ sau** (nút thứ 2, 4, ... lên đầu), giữ thứ tự tương đối. Gợi ý: sau khi gom như bài gốc, đầu danh sách mới là `even_head`.

**Bài 5 (biến thể / mini challenge).** Viết `odd_even_by_value(head)` gom nút có **giá trị lẻ** trước, **giá trị chẵn** sau, giữ thứ tự tương đối trong mỗi nhóm (chỉ đổi liên kết, `O(1)` phụ). Thử `[2,1,3,5,6,4,7]` → `[1,3,5,7,2,6,4]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: vị trí lẻ trước (`values[0::2]`), rồi vị trí chẵn. Bài 2: `even_head = 20`. Bài 4: bài gốc nhưng trả `even_head`. Bài 5: hai danh sách giả đầu.
- **Mức 2 (vừa):** Bài 1: `[1,3,5,2,4,6]`, `[9]`, `[4,3]`, `[7,9,11]` và `[8,10]`. Bài 2: `[10, 30, 20, 40]`. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: gom như bài gốc nhưng đặt `odd.next = None` (kết thúc chuỗi lẻ), rồi đi tới đuôi chuỗi chẵn và đặt `even_tail.next = head`; trả `even_head`. Bài 5: duyệt từng nút, nút giá trị lẻ gắn vào `odd_tail`, còn lại vào `even_tail`; cuối cùng `even_tail.next = None`, `odd_tail.next = even_dummy.next`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
# Bài 1
assert oracle([1, 2, 3, 4, 5, 6]) == [1, 3, 5, 2, 4, 6] and oracle([9]) == [9] and oracle([4, 3]) == [4, 3]
assert [7, 8, 9, 10, 11][0::2] == [7, 9, 11] and [7, 8, 9, 10, 11][1::2] == [8, 10]

# Bài 2
assert to_list(odd_even_two_pointers(build_list([10, 20, 30, 40]))) == [10, 30, 20, 40]


# Bài 3
def odd_even_todo_ans(head):
    if head is None or head.next is None:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head


for values, expected in examples:
    assert to_list(odd_even_todo_ans(build_list(values))) == expected


# Bài 4
def even_odd_list(head):
    if head is None or head.next is None:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = None                      # kết thúc chuỗi lẻ
    even_tail = even_head
    while even_tail.next:
        even_tail = even_tail.next
    even_tail.next = head                # chẵn trước, lẻ sau
    return even_head


assert to_list(even_odd_list(build_list([1, 2, 3, 4, 5]))) == [2, 4, 1, 3, 5]
assert to_list(even_odd_list(build_list([1, 2, 3, 4]))) == [2, 4, 1, 3]
assert to_list(even_odd_list(build_list([1]))) == [1] and to_list(even_odd_list(None)) == []


# Bài 5
def odd_even_by_value(head):
    odd_dummy, even_dummy = ListNode(), ListNode()
    o, e = odd_dummy, even_dummy
    while head:
        if head.val % 2 != 0:
            o.next = head
            o = o.next
        else:
            e.next = head
            e = e.next
        head = head.next
    e.next = None
    o.next = even_dummy.next
    return odd_dummy.next


assert to_list(odd_even_by_value(build_list([2, 1, 3, 5, 6, 4, 7]))) == [1, 3, 5, 7, 2, 6, 4]
rng = random.Random(9)
for _ in range(300):
    vals = [rng.randint(-9, 9) for _ in range(rng.randint(0, 12))]
    got = to_list(odd_even_by_value(build_list(vals)))
    assert got == [v for v in vals if v % 2 != 0] + [v for v in vals if v % 2 == 0]
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai con trỏ `odd`/`even` + `even_head`)

```python
class Solution:
    def oddEvenList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        if head is None or head.next is None:
            return head
        odd, even = head, head.next
        even_head = even
        while even and even.next:
            odd.next = even.next
            odd = odd.next
            even.next = odd.next
            even = even.next
        odd.next = even_head
        return head
```

**Checklist**

- [ ] Xử lý trước các trường hợp **rỗng** và **một nút**?
- [ ] **Lưu `even_head`** trước khi tách?
- [ ] Điều kiện dừng `while even and even.next` (không viết thiếu `even`)?
- [ ] Nối `odd.next = even_head` sau cùng?
- [ ] "Lẻ/chẵn" theo **vị trí** (bắt đầu từ 1), không phải theo **giá trị**?
- [ ] Giữ **thứ tự tương đối** trong từng nhóm; **không tạo chu trình**?
- [ ] Chỉ đổi liên kết, dùng `O(1)` bộ nhớ phụ (không chép giá trị ra mảng)?
- [ ] Skeleton đúng `class Solution` / `def oddEvenList(self, head: Optional[ListNode]) -> Optional[ListNode]`?